In [ ]:
import os
import time
import datetime

import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from scipy.ndimage import gaussian_filter
from scipy.signal import convolve2d as conv2

# Original environment: TensorFlow 1.15
# Reset the graph to avoid duplication when this notebook cell is rerun.
tf.reset_default_graph()

print('=' * 72, flush=True)
print('[READY] PSF extraction notebook initialized.', flush=True)
print('        Input files: sim_MoS2.tif, org_MoS2.tif', flush=True)
print('=' * 72, flush=True)

In [ ]:
# 1. Helper functions and fixed calculation parameters

def convol2(a, b):
    return tf.nn.conv2d(
        a,
        b,
        strides=[1, 1, 1, 1],
        padding='SAME'
    )


def save_image_cv2(
    path,
    image,
    center_normalizing=1,
    crop_fraction=0.1
):
    """Crop, normalize, and save a 2D numerical image as 16-bit TIFF.

    Parameters
    ----------
    path : str
        Output TIFF path.

    image : array-like
        Two-dimensional numerical image.

    center_normalizing : int, default=1
        1: determine the normalization min/max from the central 50% region
           of the cropped image, then apply that range to the entire cropped
           image. Values outside the reference range are clipped.
        0: determine the normalization min/max from the entire cropped image.

    crop_fraction : float, default=0.1
        Fraction removed from each side before saving. A value of 0.1 removes
        10% from the top, bottom, left, and right, retaining the central 80%
        in each dimension.

    Returns
    -------
    cropped_image : np.ndarray
        Cropped numerical image before display normalization. This returned
        array can be saved separately as TXT or NPY without intensity loss.
    """
    image = np.asarray(image, dtype=np.float64)

    if image.ndim != 2:
        raise ValueError(
            'image must be a 2D array, but shape is {}'.format(image.shape)
        )

    if center_normalizing not in (0, 1):
        raise ValueError('center_normalizing must be 0 or 1.')

    if not 0.0 <= crop_fraction < 0.5:
        raise ValueError(
            'crop_fraction must satisfy 0 <= crop_fraction < 0.5.'
        )

    height, width = image.shape
    crop_y = int(np.floor(height * crop_fraction))
    crop_x = int(np.floor(width * crop_fraction))

    y_start = crop_y
    y_end = height - crop_y if crop_y > 0 else height
    x_start = crop_x
    x_end = width - crop_x if crop_x > 0 else width

    cropped_image = image[y_start:y_end, x_start:x_end].copy()

    if cropped_image.size == 0:
        raise ValueError(
            'crop_fraction produced an empty image for shape {}.'.format(
                image.shape
            )
        )

    if center_normalizing == 1:
        cropped_height, cropped_width = cropped_image.shape

        center_y0 = cropped_height // 4
        center_y1 = cropped_height - center_y0
        center_x0 = cropped_width // 4
        center_x1 = cropped_width - center_x0

        reference_region = cropped_image[
            center_y0:center_y1,
            center_x0:center_x1
        ]
    else:
        reference_region = cropped_image

    finite_reference = reference_region[np.isfinite(reference_region)]

    if finite_reference.size == 0:
        raise ValueError('The normalization reference region has no finite values.')

    reference_min = np.min(finite_reference)
    reference_max = np.max(finite_reference)
    reference_range = reference_max - reference_min

    if reference_range > 0:
        image_view = (cropped_image - reference_min) / reference_range
    else:
        image_view = np.zeros_like(cropped_image)

    # Replace NaN/Inf before conversion and clip values outside the selected
    # normalization range.
    image_view = np.nan_to_num(
        image_view,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    image_uint16 = np.clip(
        image_view * 65535.0,
        0.0,
        65535.0
    ).astype(np.uint16)

    output_directory = os.path.dirname(path)
    if output_directory:
        os.makedirs(output_directory, exist_ok=True)

    success = cv2.imwrite(path, image_uint16)

    if not success:
        raise IOError('cv2.imwrite failed: {}'.format(path))

    return cropped_image


# Fixed calculation parameters
psfsize = 41
noise_level = 10
epoch = 100
iteration = 50
lrate = 25
eps = 1e-12
rl_output_crop_fraction = 0.1

print('=' * 72, flush=True)
print('[CONFIGURATION]', flush=True)
print('  PSF size                 : {} x {}'.format(psfsize, psfsize), flush=True)
print('  RL iterations per step   : {}'.format(iteration), flush=True)
print('  PSF optimization steps   : {}'.format(epoch), flush=True)
print('  Optimizer                : GradientDescentOptimizer', flush=True)
print('  Learning rate            : {}'.format(lrate), flush=True)
print('  PSF constraint           : non-negative and unit-sum after each update', flush=True)
print('  RL output edge crop      : {:.0f}% from each side'.format(
    100 * rl_output_crop_fraction
), flush=True)
print('  Image output             : 16-bit TIFF through save_image_cv2()', flush=True)
print('=' * 72, flush=True)

In [ ]:
# 2. Load images and reproduce the original integer-shift alignment search

print('[START] Loading and aligning input images...', flush=True)

simul = cv2.imread('sim_MoS2.tif', cv2.IMREAD_GRAYSCALE)
original = cv2.imread('org_MoS2.tif', cv2.IMREAD_GRAYSCALE)

if simul is None:
    raise FileNotFoundError('sim_MoS2.tif was not found.')
if original is None:
    raise FileNotFoundError('org_MoS2.tif was not found.')

sim = simul.astype(np.float64) / simul.max()
org = original.astype(np.float64) / original.max()

best_mse = 1.0
x = 0
y = 0

for i in range(0, 10):
    for j in range(0, 10):
        current_mse = np.average(
            np.square(
                org[i:len(org), j:len(org[i])]
                - sim[:len(org)-i, :len(org[i])-j]
            )
        )

        if best_mse > current_mse:
            best_mse = current_mse
            x = i
            y = j

orgi = org[x:len(org), y:len(org[0])]
simi = sim[0:len(sim)-x, 0:len(sim[0])-y]

print('[DONE] Image loading and alignment completed.', flush=True)
print('       Experimental image shape:', org.shape, flush=True)
print('       Simulation image shape  :', sim.shape, flush=True)
print('       Selected shift (x, y)    :', (x, y), flush=True)
print('       Alignment MSE            : {:.8e}'.format(best_mse), flush=True)
print('       Cropped common shape     :', orgi.shape, flush=True)

# All visual image output uses save_image_cv2().
save_image_cv2(
    path='org_set_a.tif',
    image=orgi,
    center_normalizing=0,
    crop_fraction=0.0
)

save_image_cv2(
    path='sim_set_a.tif',
    image=simi,
    center_normalizing=0,
    crop_fraction=0.0
)

print('[SAVED] org_set_a.tif', flush=True)
print('[SAVED] sim_set_a.tif', flush=True)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
plt.gray()
ax[0].imshow(orgi, vmin=np.min(orgi), vmax=np.max(orgi))
ax[0].set_title('Experimental image')
ax[0].axis('off')
ax[1].imshow(simi, vmin=np.min(simi), vmax=np.max(simi))
ax[1].set_title('Simulated image')
ax[1].axis('off')
plt.show()

In [ ]:
# 3. Build the differentiable RL + PSF optimization graph

print('[START] Building TensorFlow graph...', flush=True)

a = np.ones([noise_level, noise_level])

noise_filter = conv2(simi, a, mode='valid') / (noise_level * noise_level)
print('        noise_sim:', noise_filter.min(), flush=True)

sim_reference = simi - noise_filter.min()
org_reference = sim_reference / np.sum(sim_reference)

identity_psf = np.zeros((psfsize, psfsize), dtype=float)
identity_psf[psfsize // 2, psfsize // 2] = 1

orimg = tf.constant(
    org_reference[None, :, :, None],
    dtype=tf.float32
)

# Gaussian PSF retained only as an available reference tensor.
opsf = tf.constant(
    gaussian_filter(identity_psf, sigma=2)[:, :, None, None],
    dtype=tf.float32
)
opsf = opsf / tf.reduce_sum(opsf)


full = np.ones((psfsize, psfsize), dtype=np.float32)
full = full / np.sum(full)
"""
#starting value gaussian
sigma = 5

y, x = np.mgrid[
    0:psfsize,
    0:psfsize
]

center = psfsize // 2

full = np.exp(
    -(
        (x - center) ** 2
        + (y - center) ** 2
    ) / (2.0 * sigma ** 2)
).astype(np.float32)

full /= np.sum(full)


"""

rpsf = tf.Variable(
    full[:, :, None, None],
    dtype=tf.float32,
    name='rpsf'
)

noise_filter = conv2(orgi, a, mode='valid') / (noise_level * noise_level)
print('        noise_org:', noise_filter.min(), flush=True)

# Preserve the original preprocessing expression exactly.
# In Python, orgi--value is equivalent to orgi + value.
experimental_preprocessed = orgi--noise_filter.min()

blr = tf.constant(
    (
        experimental_preprocessed
        / np.sum(experimental_preprocessed)
    )[None, :, :, None],
    dtype=tf.float32
)

# Differentiable 50-iteration Richardson–Lucy restoration.
dcv = blr
for _ in range(iteration):
    estimated_blur = convol2(dcv, rpsf)
    relative_blur = blr / (estimated_blur + eps)
    dcv = dcv * convol2(relative_blur, rpsf[::-1, ::-1])

cost = tf.reduce_sum(
    tf.image.central_crop(
        tf.square(orimg - dcv),
        0.5
    )
)

# Snapshots are read before the update so no second 50-iteration RL run is needed.
cost_snapshot = tf.identity(cost, name='cost_snapshot')
psf_snapshot = tf.identity(rpsf, name='psf_snapshot')
dcv_snapshot = tf.identity(dcv, name='dcv_snapshot')

optimizer = tf.train.GradientDescentOptimizer(lrate)

with tf.control_dependencies([
    cost_snapshot,
    psf_snapshot,
    dcv_snapshot
]):
    gradient_update = optimizer.minimize(
        cost,
        var_list=[rpsf]
    )

# Project the updated PSF onto non-negative, unit-sum values.
with tf.control_dependencies([gradient_update]):
    clipped_psf = tf.maximum(rpsf, 0.0)
    clipped_sum = tf.reduce_sum(clipped_psf)
    uniform_psf = tf.ones_like(clipped_psf) / float(psfsize * psfsize)

    normalized_psf = tf.cond(
        clipped_sum > eps,
        lambda: clipped_psf / clipped_sum,
        lambda: uniform_psf
    )

    train = tf.assign(
        rpsf,
        normalized_psf,
        name='project_psf'
    )

model = tf.global_variables_initializer()

print('[DONE] TensorFlow graph built successfully.', flush=True)
print('       One outer step contains {} RL updates.'.format(iteration), flush=True)
print('       PSF is clipped and normalized after every gradient update.', flush=True)

In [ ]:
# 4. Run exactly 60 PSF optimization steps
#
# At every outer step:
#   - PSF is saved as 16-bit TIFF using full-image normalization.
#   - R–L output is cropped by 10% on every side and saved as 16-bit TIFF.
#   - The R–L TIFF uses the central 50% region for display normalization.
#   - Raw numerical matrices are saved separately as TXT.
#
# No additional RL calculation is performed for intermediate output.

print('', flush=True)
print('=' * 72, flush=True)
print('[START] PSF OPTIMIZATION STARTED', flush=True)
print('        Start time : {}'.format(datetime.datetime.now()), flush=True)
print('        Total work : {} outer steps x {} RL iterations'.format(
    epoch,
    iteration
), flush=True)
print('=' * 72, flush=True)

output_dir = 'intermediate_results_6'
psf_tif_dir = os.path.join(output_dir, 'psf_tif')
psf_txt_dir = os.path.join(output_dir, 'psf_txt')
dcv_tif_dir = os.path.join(output_dir, 'deconvolution_tif')
dcv_txt_dir = os.path.join(output_dir, 'deconvolution_txt')

for directory in (
    psf_tif_dir,
    psf_txt_dir,
    dcv_tif_dir,
    dcv_txt_dir
):
    os.makedirs(directory, exist_ok=True)

print('[OUTPUT] Intermediate results will be saved separately:', flush=True)
print('         PSF TIFF           : {}'.format(psf_tif_dir), flush=True)
print('         PSF TXT            : {}'.format(psf_txt_dir), flush=True)
print('         Deconvolution TIFF : {}'.format(dcv_tif_dir), flush=True)
print('         Deconvolution TXT  : {}'.format(dcv_txt_dir), flush=True)

run_start = time.time()
graph = np.zeros(epoch, dtype=float)

with tf.Session() as sess:
    sess.run(model)

    for step in range(epoch):
        step_number = step + 1
        step_start = time.time()

        print('', flush=True)
        print('[RUNNING] step {}/{} started at {}'.format(
            step_number,
            epoch,
            datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        ), flush=True)

        _, loss_value, psf_value, dcv_value = sess.run(
            [
                train,
                cost_snapshot,
                psf_snapshot,
                dcv_snapshot
            ]
        )

        graph[step] = loss_value

        psf_intermediate = psf_value[:, :, 0, 0]
        dcv_intermediate_full = dcv_value[0, :, :, 0]

        psf_tif_path = os.path.join(
            psf_tif_dir,
            'psf_before_update_{:02d}.tif'.format(step_number)
        )

        dcv_tif_path = os.path.join(
            dcv_tif_dir,
            'deconvolution_crop10pct_before_update_{:02d}.tif'.format(
                step_number
            )
        )

        # PSF: no crop, normalize from the full PSF image.
        psf_raw_output = save_image_cv2(
            path=psf_tif_path,
            image=psf_intermediate,
            crop_fraction=0
        )

        # R–L result: crop 10% from all sides, then normalize the TIFF using
        # the central 50% region of the cropped image.
        dcv_cropped_output = save_image_cv2(
            path=dcv_tif_path,
            image=dcv_intermediate_full,
            center_normalizing=1,
            crop_fraction=rl_output_crop_fraction
        )

        psf_txt_path = os.path.join(
            psf_txt_dir,
            'psf_before_update_{:02d}.txt'.format(step_number)
        )

        dcv_txt_path = os.path.join(
            dcv_txt_dir,
            'deconvolution_crop10pct_before_update_{:02d}.txt'.format(
                step_number
            )
        )

        np.savetxt(
            psf_txt_path,
            psf_raw_output,
            fmt='%.10e',
            header='PSF before update {}/{}; loss={:.10e}'.format(
                step_number,
                epoch,
                loss_value
            )
        )

        np.savetxt(
            dcv_txt_path,
            dcv_cropped_output,
            fmt='%.10e',
            header=(
                'Deconvolution after 10% edge crop before update {}/{}; '
                'loss={:.10e}'
            ).format(
                step_number,
                epoch,
                loss_value
            )
        )

        step_seconds = time.time() - step_start
        elapsed_seconds = time.time() - run_start
        average_seconds = elapsed_seconds / step_number
        remaining_seconds = average_seconds * (epoch - step_number)
        percent = 100.0 * step_number / epoch

        elapsed_text = str(
            datetime.timedelta(seconds=int(elapsed_seconds))
        )
        step_text = str(
            datetime.timedelta(seconds=int(step_seconds))
        )
        eta_text = str(
            datetime.timedelta(seconds=int(remaining_seconds))
        )

        print('[DONE]    step {}/{} | {:5.1f}%'.format(
            step_number,
            epoch,
            percent
        ), flush=True)
        print('          loss       : {:.8e}'.format(loss_value), flush=True)
        print('          step time  : {}'.format(step_text), flush=True)
        print('          elapsed    : {}'.format(elapsed_text), flush=True)
        print('          ETA        : {}'.format(eta_text), flush=True)
        print('          PSF TIFF   : {}'.format(psf_tif_path), flush=True)
        print('          PSF TXT    : {}'.format(psf_txt_path), flush=True)
        print('          deconv TIFF: {}'.format(dcv_tif_path), flush=True)
        print('          deconv TXT : {}'.format(dcv_txt_path), flush=True)
        print('          crop shape : {} -> {}'.format(
            dcv_intermediate_full.shape,
            dcv_cropped_output.shape
        ), flush=True)

    print('', flush=True)
    print('[FINALIZING] Reading the PSF and restored image after step 60...', flush=True)

    final_cost, final_rpsf, final_dcv, final_blr = sess.run(
        [
            cost,
            rpsf,
            dcv,
            blr
        ]
    )

print('=' * 72, flush=True)
print('[COMPLETE] PSF OPTIMIZATION FINISHED', flush=True)
print('           End time   : {}'.format(datetime.datetime.now()), flush=True)
print('           Total time : {}'.format(
    str(datetime.timedelta(seconds=int(time.time() - run_start)))
), flush=True)
print('           Final loss : {:.8e}'.format(final_cost), flush=True)
print('=' * 72, flush=True)

opsf_final = final_rpsf[:, :, 0, 0]
dcvimg_full = final_dcv[0, :, :, 0]
blrimg = final_blr[0, :, :, 0]

final_psf_tif_path = os.path.join(
    psf_tif_dir,
    'psf_after_update.tif'
)

final_dcv_tif_path = os.path.join(
    dcv_tif_dir,
    'deconvolution_crop10pct_after_update.tif'
)

final_psf_raw = save_image_cv2(
    path=final_psf_tif_path,
    image=opsf_final,
    crop_fraction=0.0
)

final_dcv_cropped = save_image_cv2(
    path=final_dcv_tif_path,
    image=dcvimg_full,
    center_normalizing=1,
    crop_fraction=rl_output_crop_fraction
)

final_psf_txt_path = os.path.join(
    psf_txt_dir,
    'psf_after_update.txt'
)

final_dcv_txt_path = os.path.join(
    dcv_txt_dir,
    'deconvolution_crop10pct_after_update.txt'
)

np.savetxt(
    final_psf_txt_path,
    final_psf_raw,
    fmt='%.10e',
    header='Final PSF after 60 updates; loss={:.10e}'.format(final_cost)
)

np.savetxt(
    final_dcv_txt_path,
    final_dcv_cropped,
    fmt='%.10e',
    header=(
        'Final deconvolution after 10% edge crop and 60 updates; '
        'loss={:.10e}'
    ).format(final_cost)
)

# Compact lossless numerical outputs for direct Python reuse.
np.save('psf_step60.npy', final_psf_raw)
np.save('deconvolution_crop10pct_step60.npy', final_dcv_cropped)
np.save('loss_60steps.npy', graph)

print('[SAVED] Final PSF TIFF   : {}'.format(final_psf_tif_path), flush=True)
print('[SAVED] Final PSF TXT    : {}'.format(final_psf_txt_path), flush=True)
print('[SAVED] Final deconv TIFF: {}'.format(final_dcv_tif_path), flush=True)
print('[SAVED] Final deconv TXT : {}'.format(final_dcv_txt_path), flush=True)
print('[CROP]  Final deconv shape: {} -> {}'.format(
    dcvimg_full.shape,
    final_dcv_cropped.shape
), flush=True)
print('[SAVED] psf_step60.npy', flush=True)
print('[SAVED] deconvolution_crop10pct_step60.npy', flush=True)
print('[SAVED] loss_60steps.npy', flush=True)

In [ ]:
# 5. Summary display and loss curve
# TIFF files were exported using save_image_cv2().

fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(16, 16))
plt.gray()

for axis in (ax[0, 0], ax[0, 1], ax[1, 0], ax[1, 1]):
    axis.axis('off')

ax[0, 0].imshow(org_reference, vmin=0, vmax=1)
ax[0, 0].set_title('Simulated reference')

ax[0, 1].imshow(
    final_dcv_cropped,
    vmin=np.min(final_dcv_cropped),
    vmax=np.max(final_dcv_cropped)
)
ax[0, 1].set_title('Deconvoluted image after step 60 (10% edge crop)')

ax[1, 0].imshow(
    blrimg,
    vmin=np.min(blrimg),
    vmax=np.max(blrimg)
)
ax[1, 0].set_title('Experimental input')

ax[1, 1].imshow(
    final_psf_raw,
    vmin=np.min(final_psf_raw),
    vmax=np.max(final_psf_raw)
)
ax[1, 1].set_title('Extracted non-negative normalized PSF after step 60')

plt.show()

plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, epoch + 1), graph, 'bo-')
plt.xlabel('PSF optimization step')
plt.ylabel('Central-crop squared-error loss')
plt.title('Optimization loss: 60 steps')
plt.grid(True)
plt.show()